E3 Novice

Brian Bannon

This notebook analyzes the kinetics and performance of three chemical reaction scenarios: pharmaceutical synthesis, specialty chemical production, and industrial bulk chemical processing. Using the Arrhenius equation and first-order reaction kinetics, it calculates reaction rate constants, conversion as a function of time, and the batch time required to reach a target conversion. The analysis illustrates how temperature, activation energy, and reaction parameters influence the efficiency and design of chemical processes.<br>The notebook also explores practical engineering considerations, such as safety margins and the effect of changing process conditions, including temperature or batch time. By combining Python functions with pandas DataFrames, the notebook provides a clear, structured framework for comparing scenarios, calculating key performance metrics, and visualizing the impact of kinetic and operational parameters on reactor outcomes.

In [11]:
import numpy as np
import pandas as pd

In [12]:
def calculate_rate_constant(temperature: float, a: float, ea: float) -> float:
    """
    Function to calculate rate constant
    :param temperature: temperature in Kelvin
    :param a: pre-exponential factor in 1/s
    :param ea: activation energy in J/mol
    :return: reaction rate constant k in 1/s
    """
    return a * np.exp(-ea / (8.314 * temperature))

In [13]:
def calculate_conversion(k: float, time: float):
    """
    Function to calculate conversion
    :param k: reaction rate constant in 1/s
    :param time: reaction time in seconds
    :return: conversion X as a fraction
    """
    return 1 - np.exp(-k * time)

In [14]:
def calculate_batch_time(k: float, target_conversion: float) -> float:
    """
    Function to calculate batch time
    :param k: reaction rate constant in 1/s
    :param target_conversion: desired conversion as a fraction
    :return: batch time in seconds
    """
    return -np.log(1 - target_conversion) / k

In [15]:
def analyze_reactor_performance(reactor_specs: dict[str, float]) -> dict[str, float]:
    """
    Function to analyze reactor performance
    :param reactor_specs: dictionary containing 'temperature', 'A', and 'Ea'
    :return: dict with analysis results
    """
    temperature = reactor_specs['temperature']
    a = reactor_specs['A']
    ea = reactor_specs['Ea']
    k = calculate_rate_constant(temperature, a, ea)

    time = reactor_specs['batch_time']
    conversion  = calculate_conversion(k, time)

    target_conversion = reactor_specs['target_conversion']
    required_time = calculate_batch_time(k, target_conversion)

    return {
        'k': k,
        'actual_conversion': conversion,
        'required_time': required_time,
        'conversion_percent': conversion * 100,
        'time_hours': time / 3600
    }

In [16]:
df = pd.DataFrame()
df['Pharmaceutical synthesis'] = [320, 2e10, 85000, 7200, 0.9]
df['Specialty chemical'] = [330, 3e8, 72000, 5400, 0.85]
df['Industrial bulk chemical'] = [340, 1.2e10, 78000, 3600, 0.95]

df.loc[len(df)] = [calculate_rate_constant(df[j][0], df[j][1], df[j][2]) for j in df]
df.loc[len(df)] = [calculate_conversion(df[j][5], df[j][3]) for j in df]
df.loc[len(df)] = [calculate_batch_time(df[j][5], df[j][4]) for j in df]
df.loc[len(df)] = [df[j][7] <= df[j][3] for j in df]
df.loc[len(df)] = [df[j][3] - df[j][7] for j in df]

df.index = ['Temperature (K)', 'Pre-exponential factor (A) (1/s)', 'Activation energy (Ea) (J/mol)', 'Batch time (h)', 'Target conversion', 'Reaction rate (k)', 'Actual conversion', 'Time required for target (h)', 'Sufficient batch time?', 'Safety margin (s)']

rows = list(df.index)

(df.style
      .format('{:,.0f}', pd.IndexSlice[:, :])
      .format(lambda x: f'{x/60/60:.1f}', pd.IndexSlice[rows[3:8:4], :])
      .format(lambda x: f'{x*100:.0f}%', pd.IndexSlice[rows[4:7:2], :])
      .format('{:.0e}', pd.IndexSlice[rows[1], :])
      .format('{:.5f}', pd.IndexSlice[rows[5], :])
      .format(lambda x: bool(x), pd.IndexSlice[rows[8], :])
      )

,Pharmaceutical synthesis,Specialty chemical,Industrial bulk chemical
Temperature (K),320,330,340
Pre-exponential factor (A) (1/s),2e+10,3e+08,1e+10
Activation energy (Ea) (J/mol),"85,000","72,000","78,000"
Batch time (h),2.0,1.5,1.0
Target conversion,90%,85%,95%
Reaction rate (k),0.00027,0.00120,0.01246
Actual conversion,85%,100%,100%
Time required for target (h),2.4,0.4,0.1
Sufficient batch time?,False,True,True
Safety margin (s),"-1,440","3,822","3,360"


In [17]:
# Question 1
df.iloc[5], min(df.iloc[5]), max(df.iloc[5]), max(df.iloc[5])/min(df.iloc[5])

(Pharmaceutical synthesis    0.000267
 Specialty chemical          0.001202
 Industrial bulk chemical    0.012460
 Name: Reaction rate (k), dtype: float64,
 0.0002665033211183028,
 0.012459697642923883,
 46.75250421135626)

In [18]:
# Question 2
a = df.iloc[1, 0]
ea = df.iloc[2, 0]
x = df.iloc[4, 0]
t = 1.5 * 60 * 60
k = -np.log(1-x)/t
r = 8.314
temp = -ea/(r * np.log(k/a))
temp

np.float64(324.7777911623967)

In [19]:
# Question 4
temp = df.iloc[0, 0] - 20
a = df.iloc[1, 0]
ea = df.iloc[2, 0]
k = df.iloc[5, 0]
kNew = calculate_rate_constant(temp, a, ea)
(kNew - k)/k*100

np.float64(-88.11557497075627)

In [20]:
# Question 5
k = df.iloc[5, 2]
t = 7200
calculate_conversion(k, t) * 100

np.float64(100.0)

QUESTION 1  
Scenario 3 has the highest reaction rate. The ratio of highest to lowest k is about 46. This shows that higher temperature and favorable kinetic parameters dramatically increase reaction rates.

QUESTION 2  
The required temperature would be about 325 K.

QUESTION 3  
Scenario 2 has the largest safety margin. Engineers might design this margin to ensure the reaction reliably reaches the target conversion despite temperature fluctuations, kinetic uncertainties, or scale-up effects.

QUESTION 4  
Scenario 1 has the highest activation energy. Reducing the temperature by 20 K drops k by about 88%, showing that high Ea reactions are very sensitive to temperature changes.

QUESTION 5  
The conversion effectively stays at 100% because the reaction was already nearly complete in 1 hour.

The analysis demonstrates how reaction kinetics and operating conditions directly impact reactor performance. Scenario 3 (industrial bulk chemical) achieves the highest reaction rate due to its combination of temperature and kinetic parameters, while Scenario 1 (pharmaceutical synthesis) is most sensitive to temperature changes because of its high activation energy. Safety margins vary across scenarios, reflecting the need to account for uncertainties in scale-up, temperature fluctuations, and kinetic variability.<br>Overall, this study highlights the importance of carefully selecting temperature, batch time, and target conversion in chemical process design. By using Python to calculate rate constants, conversions, and required batch times, engineers can optimize reactor performance, ensure reliability, and make informed decisions about process adjustments or safety buffers. This approach provides a clear, quantitative framework for evaluating and comparing different reaction scenarios.